In [7]:
!pip install lingua-language-detector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 10.9 MB/s eta 0:00:0000:0100:01


In [19]:

import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lingua import Language, LanguageDetectorBuilder
from tqdm import tqdm

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH   = 128
BATCH_SIZE   = 64
N_SAMPLES    = 1000                  
DATA_PATH    = "/kaggle/input/datasets/sanzharaltynbay/dataset-for-inference/data/reviews.csv"
RESULTS_PATH = "results/predictions.csv"

MODEL_ROUTES = {
    "kk":    "/kaggle/input/models/sanzharaltynbay/mbert-kk/transformers/default/1/mbert-kk/best_model",
    "ru":    "/kaggle/input/models/sanzharaltynbay/rubert-ru/transformers/default/1/rubert-ru/best_model",
    "other": "/kaggle/input/models/sanzharaltynbay/multilang-model/transformers/default/1/multilang_model",
}

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


In [11]:
df = pd.read_csv(DATA_PATH)

drop_cols = ['id', 'branch_id', 'rating', 'date_created', 'official_reply',
             'firm_name', 'firm_address', 'topics', 'firm_rubrics']
clear_df = df.drop(columns=[c for c in drop_cols if c in df.columns])

clear_df['text'] = clear_df['text'].astype(str).str.strip()
clear_df = clear_df[clear_df['text'].str.len() > 1]
clear_df = clear_df.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"После чистки: {clear_df.shape}")
print(clear_df.head(), "\n")

inference_df = clear_df.head(N_SAMPLES).copy()

/tmp/ipykernel_57/2650979316.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


После чистки: (957892, 1)
                                                text
0  Обсуждение 0 повара очень плохо работает как ч...
1  Все супер, все великолепно. Коллаба с андердог...
2  Обожаю Coco особенно фалафель и курицу. Классн...
3  В фри со специями есть все специи, кроме соли ...
4  Классное место, персонал вежливый  Еда очень в... 



In [13]:
languages = [Language.KAZAKH, Language.RUSSIAN, Language.ENGLISH]
detector = (
    LanguageDetectorBuilder
    .from_languages(*languages)
    .with_minimum_relative_distance(0.25)
    .build()
)
LANG_MAP = {Language.KAZAKH: "kk", Language.RUSSIAN: "ru"}

texts_clean = (
    inference_df["text"].astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
        .tolist()
)
detected = detector.detect_languages_in_parallel_of(texts_clean)
inference_df["lang_label"] = [LANG_MAP.get(l, "other") for l in detected]

print("Распределение по языкам:")
print(inference_df["lang_label"].value_counts(), "\n")


Распределение по языкам:
lang_label
ru       769
kk       217
other     14
Name: count, dtype: int64 



In [14]:
class InferenceDataset(TorchDataset):
    def __init__(self, texts, tokenizer, max_length):
        self.enc = tokenizer(
            list(texts), padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        )
    def __len__(self):
        return self.enc["input_ids"].shape[0]
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}

_model_cache = {}
def get_model(path: str):
    if path not in _model_cache:
        tok = AutoTokenizer.from_pretrained(path)
        mdl = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE).eval()
        _model_cache[path] = (tok, mdl)
    return _model_cache[path]

@torch.inference_mode()
def run_inference(texts: list[str], model_path: str):
    """Возвращает (preds, confs, id2label)."""
    if len(texts) == 0:
        return [], [], {}

    tokenizer, model = get_model(model_path)
    id2label = model.config.id2label

    loader = DataLoader(
        InferenceDataset(texts, tokenizer, MAX_LENGTH),
        batch_size=BATCH_SIZE, shuffle=False
    )

    preds, confs = [], []
    for batch in tqdm(loader, desc=f"[{os.path.basename(os.path.dirname(model_path))}]"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        probs = torch.softmax(model(**batch).logits, dim=-1)
        top_p, top_i = probs.max(dim=-1)
        preds.extend(top_i.cpu().tolist())
        confs.extend(top_p.cpu().tolist())

    return preds, confs, id2label

In [15]:
def infer_with_router(df: pd.DataFrame,
                      text_col: str = "text",
                      lang_col: str = "lang_label") -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    df["pred_label"]     = -1
    df["pred_prob"]      = -1.0
    df["pred_sentiment"] = ""

    # Список маршрутов: (значение в lang_col, ключ в MODEL_ROUTES)
    routes = [("kk", "kk"), ("ru", "ru")]

    for lang, key in routes:
        mask = df[lang_col] == lang
        if mask.sum() == 0:
            continue
        texts = df.loc[mask, text_col].tolist()
        preds, probs, id2label = run_inference(texts, MODEL_ROUTES[key])
        df.loc[mask, "pred_label"]     = np.array(preds, dtype=np.int64)
        df.loc[mask, "pred_prob"]      = np.array(probs, dtype=np.float32)
        df.loc[mask, "pred_sentiment"] = [id2label[p] for p in preds]

    # Fallback: всё что не kk и не ru
    fb_mask = ~df[lang_col].isin(["kk", "ru"])
    if fb_mask.sum() > 0:
        texts = df.loc[fb_mask, text_col].tolist()
        preds, probs, id2label = run_inference(texts, MODEL_ROUTES["other"])
        df.loc[fb_mask, "pred_label"]     = np.array(preds, dtype=np.int64)
        df.loc[fb_mask, "pred_prob"]      = np.array(probs, dtype=np.float32)
        df.loc[fb_mask, "pred_sentiment"] = [id2label[p] for p in preds]

    return df


In [ ]:
@torch.inference_mode()
def run_inference(texts, model_path):
    if len(texts) == 0:
        return [], [], {}
    tokenizer, model = get_model(model_path)
    id2label = model.config.id2label

    loader = DataLoader(InferenceDataset(texts, tokenizer, MAX_LENGTH),
                        batch_size=BATCH_SIZE, shuffle=False)
    preds, confs = [], []
    for batch in tqdm(loader, desc=f"Inference [{os.path.basename(model_path)}]"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1)
        top_p, top_i = probs.max(dim=-1)
        preds.extend(top_i.cpu().tolist())
        confs.extend(top_p.cpu().tolist())
    return preds, confs, id2label

In [5]:
def infer_with_router(df, text_col="text", lang_col="lang_label"):
    df = df.copy().reset_index(drop=True)
    df["pred_label"] = -1
    df["pred_prob"]  = -1.0
    df["pred_sentiment"] = ""

    routes = [("kk","kk"), ("ru","ru")]
    fallback_mask = ~df[lang_col].isin(["kk","ru"])

    for lang, key in routes:
        mask = df[lang_col] == lang
        if mask.sum() == 0: continue
        texts = df.loc[mask, text_col].tolist()
        preds, probs, id2label = run_inference(texts, MODEL_ROUTES[key])
        df.loc[mask, "pred_label"] = np.array(preds, dtype=np.int64)
        df.loc[mask, "pred_prob"]  = np.array(probs, dtype=np.float32)
        df.loc[mask, "pred_sentiment"] = [id2label[p] for p in preds]

    if fallback_mask.sum() > 0:
        texts = df.loc[fallback_mask, text_col].tolist()
        preds, probs, id2label = run_inference(texts, MODEL_ROUTES["other"])
        df.loc[fallback_mask, "pred_label"] = np.array(preds, dtype=np.int64)
        df.loc[fallback_mask, "pred_prob"]  = np.array(probs, dtype=np.float32)
        df.loc[fallback_mask, "pred_sentiment"] = [id2label[p] for p in preds]
    return df

In [20]:
print(f"Total samples: {len(inference_df)}\n")
results_df = infer_with_router(inference_df)

results_df.to_csv(RESULTS_PATH, index=False)
print(f"\n✅ Saved to {RESULTS_PATH}")

Total samples: 1000



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[mbert-kk]: 100%|██████████| 4/4 [00:01<00:00,  2.53it/s]


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

[rubert-ru]: 100%|██████████| 13/13 [00:00<00:00, 38.24it/s]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[1]: 100%|██████████| 1/1 [00:00<00:00, 19.75it/s]


✅ Saved to results/predictions.csv
